In [17]:
%pip install pandas click datasets

Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd
from collections import defaultdict
import numpy as np

def load_sentiwordnet(path):
    scores = defaultdict(list)

    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.startswith("#"):
                continue

            parts = line.strip().split('\t')
            if len(parts) != 6:
                continue

            pos, synset_id, pos_score, neg_score, terms, gloss = parts
            pos_score = float(pos_score)
            neg_score = float(neg_score)

            words = terms.split()

            for w in words:
                word = w.split('#')[0]
                scores[word].append((pos_score, neg_score))


    # average scores per word
    word_scores = {}
    for word, vals in scores.items():
        pos_avg = sum(v[0] for v in vals) / len(vals)
        neg_avg = sum(v[1] for v in vals) / len(vals)
        word_scores[word] = (pos_avg, neg_avg)

    return word_scores


swn = load_sentiwordnet("SentiWordNet_3.0.0.txt")

print(swn["cheap"])
print(swn["good"])
print(swn["bad"])

(0.0625, 0.4375)
(0.5694444444444444, 0.004629629629629629)
(0.029411764705882353, 0.625)


In [14]:
def rank_words_by_connotation(swn_scores):
    """
    Build a sorted list of words with positive, negative, and total connotation score.

    total_connotation = positive_score + negative_score
    """
    ranked = []

    for word, (pos_score, neg_score) in swn_scores.items():
        #total_score = pos_score + neg_score
        total_score = pos_score*neg_score
        ranked.append((word, pos_score, neg_score, total_score))
        

    ranked.sort(key=lambda x: x[3], reverse=True)
    return ranked


ranked_words = rank_words_by_connotation(swn)

def print_top_words(ranked_words, top_k=10):
    print("Top 10 words by total connotation score:")
    for word, pos_score, neg_score, total_score in ranked_words[:top_k]:
        print(f"{word:20s} pos={pos_score:.3f} neg={neg_score:.3f} total={total_score:.3f}")
  

print(swn["cheap"])

(0.0625, 0.4375)


In [8]:

def store_word_list(word_list, filename):
    #first recover wordd already in the file if it exists
    try:        
        with open(filename, 'r', encoding='utf-8') as f:
            existing_words = set(line.strip() for line in f)
    except FileNotFoundError:
        existing_words = set()
    #combine existing words with new ones, avoiding duplicates
    combined_words = existing_words.union(set(word_list))
    with open(filename, 'w', encoding='utf-8') as f:
        for word in combined_words:
            f.write(word + '\n')

word_list = [w[0] for w in ranked_words[:20]]
store_word_list(word_list, "top_connotation_words.txt")

In [52]:
from collections import Counter
import re

from datasets import load_dataset

# Load only SST2 train split from Hugging Face Datasets
sst = load_dataset("sst2", split="train")

word_counts = Counter()

if "sentence" in sst.column_names:
    sentence_field = "sentence"
elif "text" in sst.column_names:
    sentence_field = "text"
else:
    raise ValueError("No sentence/text field found in SST2 train split.")

for sentence in sst[sentence_field]:
    words = re.findall(r"[a-z']+", sentence.lower())
    word_counts.update(words)

# Sorted list of (word, count) from most frequent to least frequent
sorted_words_by_occurrence = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)

In [53]:
#print the number of words that have more than 100 occurrences
num_words_over_100 = sum(1 for word, count in sorted_words_by_occurrence if count > 100)
print(f"Number of words with more than 100 occurrences: {num_words_over_100}")

#extract the words in swn that have more than 100 occurrences in SST
frequent_words_in_sst = [word for word, count in sorted_words_by_occurrence if count > 200 and word in swn]
# rank the frequent words in sst by their total connotation score in swn
frequent_words_in_sst_ranked = sorted(frequent_words_in_sst, key=lambda w: swn[w][0]*swn[w][1], reverse=True)

word_base = [word for word in frequent_words_in_sst_ranked[:50]]

print("Top frequent words in SST ranked by connotation score:")
for word in word_base:
    pos_score, neg_score = swn[word]
    total_score = pos_score*neg_score
    print(f"{word:20s} pos={pos_score:.3f} neg={neg_score:.3f} total={total_score:.3f}")
    

Number of words with more than 100 occurrences: 702
Top frequent words in SST ranked by connotation score:
might                pos=0.375 neg=0.500 total=0.188
entertaining         pos=0.625 neg=0.250 total=0.156
pretty               pos=0.333 neg=0.417 total=0.139
enjoyable            pos=0.500 neg=0.250 total=0.125
quite                pos=0.312 neg=0.312 total=0.098
smart                pos=0.306 neg=0.222 total=0.068
worst                pos=0.083 neg=0.688 total=0.057
charming             pos=0.438 neg=0.125 total=0.055
emotional            pos=0.406 neg=0.125 total=0.051
rather               pos=0.156 neg=0.281 total=0.044
fresh                pos=0.144 neg=0.250 total=0.036
very                 pos=0.375 neg=0.094 total=0.035
funny                pos=0.125 neg=0.275 total=0.034
strong               pos=0.225 neg=0.150 total=0.034
simply               pos=0.344 neg=0.094 total=0.032
compelling           pos=0.250 neg=0.125 total=0.031
other                pos=0.094 neg=0.312 tota

In [ ]:
# Build a dataset with columns: sentence | word | label
# Keep only words that are present in word_base.
word_base_set = set(word_base)
rows = []

if "sentence" in sst.column_names:
    sentence_field = "sentence"
elif "text" in sst.column_names:
    sentence_field = "text"
else:
    raise ValueError("No sentence/text field found in SST2 train split.")

has_label = "label" in sst.column_names

for example in sst:
    sentence = example[sentence_field]
    label = example["label"] if has_label else None

    words = re.findall(r"[a-z']+", sentence.lower())
    for word in words:
        if word in word_base_set:
            rows.append(
                {
                    "sentence": sentence,
                    "word": word,
                    "label": label,
                }
            )

sst2_sentence_word_label = pd.DataFrame(rows, columns=["sentence", "word", "label"])

print(f"Constructed rows: {len(sst2_sentence_word_label)}")
sst2_sentence_word_label.head(10)

Constructed rows: 23949


,sentence,word,label
0,hide new secretions from the parental units,new,0
1,"contains no wit , only labored gags",no,0
2,that loves its characters and communicates som...,rather,1
3,on the worst revenge-of-the-nerds clichés the ...,worst,0
4,demonstrates that the director of such hollywo...,still,1
5,demonstrates that the director of such hollywo...,emotional,1
6,a depressed fifteen-year-old 's suicidal poetry,old,0
7,"the part where nothing 's happening ,",nothing,0
8,saw how bad this movie was,bad,0
9,lend some dignity to a dumb story,some,0


In [ ]:
example_word = "rather"

examples = sst2_sentence_word_label[sst2_sentence_word_label["word"] == example_word]

for idx, row in examples.iterrows():
    print(f"Sentence: {row['sentence']}")
    print(f"Label: {row['label']}")
    print()
    

Mean label for the word 'rather': 0.340
Sentence: that loves its characters and communicates something rather beautiful about human nature 
Label: 1

Sentence: with color and depth , and rather a good time 
Label: 1

Sentence: rather choppy 
Label: 0

Sentence: ... familiar and predictable , and 4/5ths of it might as well have come from a xerox machine rather than ( writer-director ) franc . 
Label: 0

Sentence: famuyiwa 's feature deals with its subject matter in a tasteful , intelligent manner , rather than forcing us to endure every plot contrivance that the cliché-riddled genre can offer . 
Label: 1

Sentence: a watch that makes time go faster rather than 
Label: 1

Sentence: like one long , meandering sketch inspired by the works of john waters and todd solondz , rather than a fully developed story 
Label: 0

Sentence: a rather simplistic one : grief drives her , love drives him , and a second chance to find love in the most unlikely place 
Label: 1

Sentence: a laughable -- or ra